# Exercise 03.07 (Assignment #01) — Bases, Projections and Approximations

In [1]:
import numpy as np

## Background

- A vector `v` is **independent** of vectors `a, b, c` if there's no solution to `v = μa + νb + ωc`. If a solution exists, the collection is **linearly dependent**.
- In `D` dimensions, a collection of independent vectors has at most `D` vectors.
- A set of `n` independent vectors spans an `n`-dimensional **subspace**.

## Problem 1 — Find a basis

Write a program that looks at a collection (list) of vectors of length `D` and finds all possible bases, if any, considering the exceptions:
- "There are less than D independent vectors in the set."
- "The list of vectors contains vectors of different dimensions."

Example: for `L = [[1,2,3,4],[5,6,7,8],[6,8,10,12],[4,4,4,4],[3,2,1,0]]`, the program should print `"There are less than D independent vectors in the set."`

In [2]:
from itertools import combinations

def find_bases(L):
    D = len(L[0])
    if any(len(v) != D for v in L):
        print("The list of vectors contains vectors of different dimensions.")
        return

    # A basis is D vectors that are independent, i.e. the matrix of those vectors has rank D
    bases = [b for b in combinations(L, D) if np.linalg.matrix_rank(np.array(b)) == D]

    if not bases:
        print("There are less than D independent vectors in the set.")
        return
    for b in bases:
        print([list(v) for v in b])

L = [[1,2,3,4],[5,6,7,8],[6,8,10,12],[4,4,4,4],[3,2,1,0]]
find_bases(L)

print()
find_bases([[1,2,3],[4,5]])                    # different dimensions
print()
find_bases([[1,0,0],[0,1,0],[0,0,1],[1,1,1]])  # has bases

There are less than D independent vectors in the set.

The list of vectors contains vectors of different dimensions.

[[1, 0, 0], [0, 1, 0], [0, 0, 1]]
[[1, 0, 0], [0, 1, 0], [1, 1, 1]]
[[1, 0, 0], [0, 0, 1], [1, 1, 1]]
[[0, 1, 0], [0, 0, 1], [1, 1, 1]]


## Problem 2 — Projection onto a vector

The parallel component of `b` projected onto `a` is:

$$\hat{a} = \frac{a}{\|a\|}, \qquad b_{\parallel} = (\hat{a} \cdot b)\,\hat{a}$$

Write a program that calculates the projection of a vector over another, and returns the parallel component.

Example: for `v = [1,2,3,4]` and `w = [0,0,1,0]`, the program should print `"[0,0,3,0]"`.

In [3]:
def fmt(v):
    # print a vector like [0,0,3,0]
    v = np.round(v, 10) + 0
    return "[" + ",".join(str(int(x)) if x == int(x) else str(x) for x in v) + "]"

def project(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    a_hat = a / np.linalg.norm(a)      # direction of a
    return (a_hat @ b) * a_hat         # parallel component of b over a

v = [1,2,3,4]
w = [0,0,1,0]
print(fmt(project(w, v)))

[0,0,3,0]


## Problem 3 — Projection onto a subspace

Given an orthonormal basis `{e1, ..., en}` for a subspace `W`, the projection of `v` onto `W` is:

$$v_W = (v \cdot e_1)e_1 + \cdots + (v \cdot e_n)e_n$$

Write a program that calculates the projection of a vector over a subspace generated by a list of given orthonormal vectors, and returns the parallel component.

Example: for the spanning set `L = [[1/√2, 1/√2, 0, 0], [1/√2, -1/√2, 0, 0], [1/√3, 1/√3, 0, 1/√3]]` and `v = [2,3,4,5]`, the program should print `"[2,3,0,5]"`.

In [4]:
def gram_schmidt(L):
    # Make the list orthonormal: remove what lies along earlier vectors, then scale to length 1
    basis = []
    for v in np.array(L, dtype=float):
        w = v - sum((v @ e) * e for e in basis)
        if np.linalg.norm(w) > 1e-10:            # skip vectors already in the span
            basis.append(w / np.linalg.norm(w))
    return basis

def project_subspace(L, v):
    v = np.array(v, dtype=float)
    return sum((v @ e) * e for e in gram_schmidt(L))   # (v.e1)e1 + ... + (v.en)en

s2, s3 = np.sqrt(2), np.sqrt(3)
L = [[1/s2, 1/s2, 0, 0], [1/s2, -1/s2, 0, 0], [1/s3, 1/s3, 0, 1/s3]]
v = [2,3,4,5]
print(fmt(project_subspace(L, v)))

[2,3,0,5]


Note: the three vectors in the example are not actually orthonormal (`e1 · e3 ≠ 0`). Using the formula on them directly gives a wrong answer, so `gram_schmidt` makes them orthonormal first, as the exercise describes. The result `[2,3,0,5]` is the projection onto the same subspace.